In [15]:
def rot32(x, n):
    """
    Left-rotate a 32-bit integer x by n bit positions
    """
    return ((x << n) | (x >> (32 - n))) & 0xFFFFFFFF


def quarter_round(a, b, c, d):
    """
    Apply the ChaCha20 quarter-round transformation to four 32-bit words
    All additions are modulo 2^32
    If result exceeds 32 bits only the least significant 32 bits are kept
    """
    # Operation 1
    a = (a + b) & 0xFFFFFFFF   # modular addition
    d = d ^ a                  # XOR
    d = rot32(d, 16)           # left rotation by 16, changed to 15 for Task 7

    # Operation 2
    c = (c + d) & 0xFFFFFFFF
    b = b ^ c
    b = rot32(b, 12)

    # Operation block 3
    a = (a + b) & 0xFFFFFFFF
    d = d ^ a
    d = rot32(d, 8)

    # Operation block 4
    c = (c + d) & 0xFFFFFFFF
    b = b ^ c
    b = rot32(b, 7)

    return a, b, c, d

def count_differing_bits(x, y):
    """
    Count the number of bit positions where x and y differ
    by XORing them (1 wherever they differ) then counting the 1s

    Used for Task 6 and Task 7
    """
    return bin(x ^ y).count('1')

def quarter_round_staged(a, b, c, d):
    """
    Same transformation as quarter_round but returns the intermediate
    state (a, b, c, d) after each of the four operation blocks so that
    diffusion can be measured as it accumulates across the function.
    
    Used for Task 9
    """
    # Operation block 1
    a = (a + b) & 0xFFFFFFFF
    d = d ^ a
    d = rot32(d, 16)
    state_after_block1 = (a, b, c, d)
 
    # Operation block 2
    c = (c + d) & 0xFFFFFFFF
    b = b ^ c
    b = rot32(b, 12)
    state_after_block2 = (a, b, c, d)
 
    # Operation block 3
    a = (a + b) & 0xFFFFFFFF
    d = d ^ a
    d = rot32(d, 8)
    state_after_block3 = (a, b, c, d)
 
    # Operation block 4
    c = (c + d) & 0xFFFFFFFF
    b = b ^ c
    b = rot32(b, 7)
    state_after_block4 = (a, b, c, d)
 
    return state_after_block1, state_after_block2, state_after_block3, state_after_block4

In [ ]:
a, b, c, d = quarter_round(
    0x11111111, # a
    0x01020304, # b
    0x9b8d6f43, # c
    0x01234567) # d

print(f"a = {hex(a)}")
print(f"b = {hex(b)}")
print(f"c = {hex(c)}")
print(f"d = {hex(d)}")

a = 0xea2a92f4
b = 0xcb1cf8ce
c = 0x4581472e
d = 0x5881c4bb


In [5]:
a, b, c, d = quarter_round(
    0x11111101, # a
    0x01020304, # b
    0x9b8d6f43, # c
    0x01234567) # d

print(f"a = {hex(a)}")
print(f"b = {hex(b)}")
print(f"c = {hex(c)}")
print(f"d = {hex(d)}")

a = 0xea2a92e3
b = 0xb3147876
c = 0x3571562e
d = 0x4881d3bb


In [7]:
a, b, c, d = quarter_round(
    0x11111111, # a
    0x01020304, # b
    0x9b8d6f43, # c
    0x01234567) # d

e, f, g, h = quarter_round(
    0x11111101, # a
    0x01020304, # b
    0x9b8d6f43, # c
    0x01234567) # d

print(f"a = {bin(a)}")
print(f"a'= {bin(e)}\n")
print(f"b = {bin(b)}")
print(f"b'= {bin(f)}\n")
print(f"c = {bin(c)}")
print(f"c'= {bin(g)}\n")
print(f"d = {bin(d)}")
print(f"d'= {bin(h)}")

a = 0b11101010001010101001001011110100
a'= 0b11101010001010101001001011100011

b = 0b11001011000111001111100011001110
b'= 0b10110011000101000111100001110110

c = 0b1000101100000010100011100101110
c'= 0b110101011100010101011000101110

d = 0b1011000100000011100010010111011
d'= 0b1001000100000011101001110111011


In [12]:
# Task 6
# and Task 7

# ── Original inputs ──────────────────────────────────────────────────────────
A = 0x11111111
B = 0x01020304
C = 0x9b8d6f43
D = 0x01234567
 
orig_a, orig_b, orig_c, orig_d = quarter_round(A, B, C, D)
 
print("=" * 64)
print("Task 6: Avalanche Effect Analysis (Modified for Task 7 with 15 rotation)")
print("Flip each of the 32 bit positions of input 'a' one at a time")
print("=" * 64)
print(f"\nOriginal inputs:  a={hex(A)}  b={hex(B)}  c={hex(C)}  d={hex(D)}")
print(f"Original outputs: a={hex(orig_a)}  b={hex(orig_b)}  "
      f"c={hex(orig_c)}  d={hex(orig_d)}\n")
 
print(f"{'Bit flipped':>12} | {'a bits':>6} | {'b bits':>6} | "
      f"{'c bits':>6} | {'d bits':>6} | {'Total':>6}")
print("-" * 64)
 
total_bits_changed = []
 
for bit in range(32):
    # Flip bit position 'bit' in input a
    a_flipped = A ^ (1 << bit)
 
    # Run quarter-round with the modified input
    mod_a, mod_b, mod_c, mod_d = quarter_round(a_flipped, B, C, D)
 
    # Count differing bits in each output word
    diff_a = count_differing_bits(orig_a, mod_a)
    diff_b = count_differing_bits(orig_b, mod_b)
    diff_c = count_differing_bits(orig_c, mod_c)
    diff_d = count_differing_bits(orig_d, mod_d)
    total  = diff_a + diff_b + diff_c + diff_d
 
    total_bits_changed.append(total)
 
    print(f"{'bit ' + str(bit):>12} | {diff_a:>6} | {diff_b:>6} | "
          f"{diff_c:>6} | {diff_d:>6} | {total:>6}")
 
average = sum(total_bits_changed) / len(total_bits_changed)
minimum = min(total_bits_changed)
maximum = max(total_bits_changed)
 
print("-" * 64)
print(f"\nSummary across all 32 single-bit flips of input 'a':")
print(f"  Average bits changed : {average:.2f} / 128")
print(f"  Minimum bits changed : {minimum}")
print(f"  Maximum bits changed : {maximum}")

Task 6: Avalanche Effect Analysis (Modified for Task 7 with 15 rotation)
Flip each of the 32 bit positions of input 'a' one at a time

Original inputs:  a=0x11111111  b=0x1020304  c=0x9b8d6f43  d=0x1234567
Original outputs: a=0xea2a92f4  b=0xcb1cf8ce  c=0x4581472e  d=0x5881c4bb

 Bit flipped | a bits | b bits | c bits | d bits |  Total
----------------------------------------------------------------
       bit 0 |      4 |     17 |     10 |      5 |     36
       bit 1 |      4 |     16 |     15 |      5 |     40
       bit 2 |      5 |     14 |      8 |      7 |     34
       bit 3 |      4 |     14 |      8 |      5 |     31
       bit 4 |      4 |     10 |      9 |      5 |     28
       bit 5 |      6 |     12 |     11 |      7 |     36
       bit 6 |      4 |     11 |     10 |      5 |     30
       bit 7 |      4 |     12 |     10 |      5 |     31
       bit 8 |      3 |     13 |      9 |      6 |     31
       bit 9 |      6 |      7 |      6 |      7 |     26
      bit 10 |   

Conclusion:
On average a single-bit flip in 'a' changes 28.09 of 128 output bits (21.9%). This demonstrates strong diffusion in the ChaCha quarter-round.

In [11]:
A = 0x11111111
B = 0x01020304
C = 0x9b8d6f43
D = 0x01234567
 
orig = quarter_round(A, B, C, D)
orig_a, orig_b, orig_c, orig_d = orig
 
print("=" * 68)
print("Task 8: Input Word Influence Analysis")
print("Flip bit 0 of each input word in turn, hold all others constant")
print("=" * 68)
 
print(f"\nOriginal inputs:  a={hex(A)}  b={hex(B)}  c={hex(C)}  d={hex(D)}")
print(f"Original outputs: a={hex(orig_a)}  b={hex(orig_b)}  "
      f"c={hex(orig_c)}  d={hex(orig_d)}\n")
 
# Each experiment: (label, modified inputs)
experiments = [
    ("a", A ^ 1, B,     C,     D    ),
    ("b", A,     B ^ 1, C,     D    ),
    ("c", A,     B,     C ^ 1, D    ),
    ("d", A,     B,     C,     D ^ 1),
]
 
print(f"{'Word flipped':>14} | {'out-a bits':>10} | {'out-b bits':>10} | "
      f"{'out-c bits':>10} | {'out-d bits':>10} | {'Total':>6} | {'All 4 affected?':>15}")
print("-" * 90)
 
for label, a, b, c, d in experiments:
    mod_a, mod_b, mod_c, mod_d = quarter_round(a, b, c, d)
 
    diff_a = count_differing_bits(orig_a, mod_a)
    diff_b = count_differing_bits(orig_b, mod_b)
    diff_c = count_differing_bits(orig_c, mod_c)
    diff_d = count_differing_bits(orig_d, mod_d)
    total  = diff_a + diff_b + diff_c + diff_d
    all_affected = "Yes" if all(x > 0 for x in [diff_a, diff_b, diff_c, diff_d]) else "No"
 
    print(f"{'bit 0 of ' + label:>14} | {diff_a:>10} | {diff_b:>10} | "
          f"{diff_c:>10} | {diff_d:>10} | {total:>6} | {all_affected:>15}")
 
    # Print the binary comparison for each output word
    print(f"\n  Flipping bit 0 of {label}:")
    for word_label, orig_val, mod_val in [
        ("a", orig_a, mod_a),
        ("b", orig_b, mod_b),
        ("c", orig_c, mod_c),
        ("d", orig_d, mod_d),
    ]:
        orig_bits = f"{orig_val:032b}"
        mod_bits  = f"{mod_val:032b}"
        diff_count = count_differing_bits(orig_val, mod_val)
        # Mark positions that differ with ^
        markers = "".join("^" if o != m else " " for o, m in zip(orig_bits, mod_bits))
        prefix = f"    {word_label}  = "          # e.g. "    a  = " — 8 chars
        print(f"{prefix}{orig_bits}")
        print(f"    {word_label}' = {mod_bits}  ({diff_count} bits changed)")
        print(f"{' ' * len(prefix)}{markers}")
    print()
 
print("-" * 90)

Task 8: Input Word Influence Analysis
Flip bit 0 of each input word in turn, hold all others constant

Original inputs:  a=0x11111111  b=0x1020304  c=0x9b8d6f43  d=0x1234567
Original outputs: a=0xea2a92f4  b=0xcb1cf8ce  c=0x4581472e  d=0x5881c4bb

  Word flipped | out-a bits | out-b bits | out-c bits | out-d bits |  Total | All 4 affected?
------------------------------------------------------------------------------------------
    bit 0 of a |          4 |         17 |         10 |          5 |     36 |             Yes

  Flipping bit 0 of a:
    a  = 11101010001010101001001011110100
    a' = 00111010001010101001001011010100  (4 bits changed)
         ^^ ^                      ^     
    b  = 11001011000111001111100011001110
    b' = 01001010100011000000111100110111  (17 bits changed)
         ^      ^^  ^    ^^^^ ^^^^^^^^  ^
    c  = 01000101100000010100011100101110
    c' = 01000110100000100110011011011110  (10 bits changed)
               ^^      ^^  ^    ^^^^^    
    d  = 010110

Conclusion:
A single-bit flip in any input word propagates to all four output words. This is because the quarter-round chains every word into every other word through its four ARX blocks — no word is ever updated in isolation.

In [ ]:
# Task 9

# Original inputs
A  = 0x11111111
B  = 0x01020304
C  = 0x9b8d6f43
D  = 0x01234567
 
# Modified input: bit 0 of a flipped
A_MOD = A ^ 1
 
# Existing Task 4 output (unchanged)
a, b, c, d = quarter_round(A,     B, C, D)
e, f, g, h = quarter_round(A_MOD, B, C, D)
 
print(f"a = {a:032b}")
print(f"a'= {e:032b}\n")
print(f"b = {b:032b}")
print(f"b'= {f:032b}\n")
print(f"c = {c:032b}")
print(f"c'= {g:032b}\n")
print(f"d = {d:032b}")
print(f"d'= {h:032b}")
 
# Task 9: Reduced Transformation Analysis
print("\n" + "=" * 68)
print("Task 9: Reduced Transformation Analysis")
print("Observe how diffusion grows after each operation block")
print(f"Original a={hex(A)}  Modified a={hex(A_MOD)}  (bit 0 flipped)")
print("=" * 68)
 
orig_stages = quarter_round_staged(A,     B, C, D)
mod_stages  = quarter_round_staged(A_MOD, B, C, D)
 
block_labels = [
    "Block 1  a+=b; d^=a; d<<<16",
    "Block 2  c+=d; b^=c; b<<<12",
    "Block 3  a+=b; d^=a; d<<<8 ",
    "Block 4  c+=d; b^=c; b<<<7 ",
]
 
for i, (label, orig_state, mod_state) in enumerate(
        zip(block_labels, orig_stages, mod_stages), start=1):
 
    diffs        = [count_differing_bits(orig_state[j], mod_state[j]) for j in range(4)]
    total        = sum(diffs)
    words_hit    = sum(1 for x in diffs if x > 0)
 
    print(f"\nAfter {label}")
    print(f"  State (original): a={hex(orig_state[0])}  b={hex(orig_state[1])}  "
          f"c={hex(orig_state[2])}  d={hex(orig_state[3])}")
    print(f"  State (modified): a={hex(mod_state[0])}  b={hex(mod_state[1])}  "
          f"c={hex(mod_state[2])}  d={hex(mod_state[3])}")
    print(f"  Bits changed — a:{diffs[0]}  b:{diffs[1]}  c:{diffs[2]}  d:{diffs[3]}"
          f"  |  total: {total}/128  |  words affected: {words_hit}/4")
 
    # Binary diff for each word
    for word_label, ov, mv in zip("abcd", orig_state, mod_state):
        orig_bits = f"{ov:032b}"
        mod_bits  = f"{mv:032b}"
        markers   = "".join("^" if o != m else " " for o, m in zip(orig_bits, mod_bits))
        prefix    = f"    {word_label}  = "
        print(f"{prefix}{orig_bits}")
        print(f"    {word_label}' = {mod_bits}")
        print(f"{' ' * len(prefix)}{markers}")

a = 11101010001010101001001011110100
a'= 00111010001010101001001011010100

b = 11001011000111001111100011001110
b'= 01001010100011000000111100110111

c = 01000101100000010100011100101110
c'= 01000110100000100110011011011110

d = 01011000100000011100010010111011
d'= 01011001100000011110010001101011

Task 9: Reduced Transformation Analysis
Observe how diffusion grows after each operation block
Original a=0x11111111  Modified a=0x11111110  (bit 0 flipped)

After Block 1  a+=b; d^=a; d<<<16
  State (original): a=0x12131415  b=0x1020304  c=0x9b8d6f43  d=0x51721330
  State (modified): a=0x12131414  b=0x1020304  c=0x9b8d6f43  d=0x51731330
  Bits changed — a:1  b:0  c:0  d:1  |  total: 2/128  |  words affected: 2/4
    a  = 00010010000100110001010000010101
    a' = 00010010000100110001010000010100
                                        ^
    b  = 00000001000000100000001100000100
    b' = 00000001000000100000001100000100
                                         
    c  = 1001101110001101011011